In [1]:
import os
import re
import torch
import cv2
import numpy as np

RAW_DATA_DIR = './hard_streaming'  

# ==========================================
# تنظیمات جدید برای استخراج ویدیوهای متفاوت
# ==========================================
SKIP_PREVIOUS = 2   # تعداد ویدیوهایی که قبلاً ساختید (برای پرش از آن‌ها)
TARGET_COUNT = 6    # تعداد ویدیوهای جدیدی که الان می‌خواهید استخراج کنید

accident_seen = 0
normal_seen = 0
accident_saved = 0
normal_saved = 0

print(f"در حال پرش از {SKIP_PREVIOUS} فایل اول و استخراج {TARGET_COUNT} ویدیوی جدید...")

for filename in os.listdir(RAW_DATA_DIR):
    if not filename.endswith('.pt'):
        continue
        
    # اگر به تعداد مورد نظر رسیدیم، حلقه متوقف می‌شود
    if accident_saved >= TARGET_COUNT and normal_saved >= TARGET_COUNT:
        break
        
    match = re.search(r'label_(\d+)', filename)
    if not match:
        continue
        
    label = int(match.group(1))
    
    # لاجیک استخراج و پرش برای تصادف‌ها
    if label == 1:
        accident_seen += 1
        if accident_seen <= SKIP_PREVIOUS:
            continue  # از فایل‌هایی که قبلا تبدیل کردی رد شو
        if accident_saved < TARGET_COUNT:
            accident_saved += 1
            out_name = f'accident_demo_{accident_seen}.mp4'
        else:
            continue
            
    # لاجیک استخراج و پرش برای عادی‌ها
    elif label == 0:
        normal_seen += 1
        if normal_seen <= SKIP_PREVIOUS:
            continue  # از فایل‌هایی که قبلا تبدیل کردی رد شو
        if normal_saved < TARGET_COUNT:
            normal_saved += 1
            out_name = f'normal_demo_{normal_seen}.mp4'
        else:
            continue
            
    try:
        # لود کردن فایل خام
        file_path = os.path.join(RAW_DATA_DIR, filename)
        frames = torch.load(file_path, weights_only=True)
        
        # مطمئن شویم فایل خام تصویری است (۴ بعدی)
        if frames.ndimension() == 4:
            if frames.size(1) == 3:
                frames = frames.permute(0, 2, 3, 1)
                
            frames_np = frames.numpy()
            
            # تبدیل مقادیر به استاندارد پیکسلی
            if frames_np.dtype != np.uint8:
                if frames_np.max() <= 1.0:
                    frames_np = (frames_np * 255).astype(np.uint8)
                else:
                    frames_np = frames_np.astype(np.uint8)
                    
            N, H, W, C = frames_np.shape
            
            # ساخت ویدیو
            out = cv2.VideoWriter(out_name, cv2.VideoWriter_fourcc(*'mp4v'), 5, (W, H))
            
            for img in frames_np:
                bgr_img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
                out.write(bgr_img)
            out.release()
            
            print(f"✅ ویدیوی جدید {out_name} با موفقیت ساخته شد (فایل منبع: {filename})")
    except Exception as e:
        print(f"خطا در ساخت ویدیو از {filename}: {e}")

print("\n🎉 ویدیوهای جدید در کنار نوت‌بوک آماده هستند!")

در حال پرش از 2 فایل اول و استخراج 6 ویدیوی جدید...
✅ ویدیوی جدید accident_demo_3.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_101_label_1.pt)
✅ ویدیوی جدید accident_demo_4.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_102_label_1.pt)
✅ ویدیوی جدید accident_demo_5.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_103_label_1.pt)
✅ ویدیوی جدید accident_demo_6.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_104_label_1.pt)
✅ ویدیوی جدید accident_demo_7.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_105_label_1.pt)
✅ ویدیوی جدید accident_demo_8.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_106_label_1.pt)
✅ ویدیوی جدید normal_demo_3.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_752_label_0.pt)
✅ ویدیوی جدید normal_demo_4.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_753_label_0.pt)
✅ ویدیوی جدید normal_demo_5.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_754_label_0.pt)
✅ ویدیوی جدید normal_demo_6.mp4 با موفقیت ساخته شد (فایل منبع: hard_video_755_label_0.pt)
✅ ویدیوی جدید normal_demo_7.mp4 با م

ba in file mitoonim video haye khodemoon ro az dataset begirim